<a href="https://colab.research.google.com/github/suprematis/systems-coding-lab/blob/main/notebooks/scratchpad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To solve this without manually guessing pixel coordinates in text, you can build an **interactive ROI calibration tool directly inside a Jupyter Notebook** using `ipywidgets` and `matplotlib`.

Instead of forcing a rigid mathematical grid, you define **base template boxes** and use interactive sliders or click handlers to adjust individual stall offsets, widths, and heights in real time, then dump the exact coordinates to a reusable JSON configuration file.

---

### Step 1: Install Dependencies

Make sure your Jupyter environment has interactive widget support enabled:

```bash
pip install ipywidgets matplotlib pillow pandas

```

---

### Step 2: The Interactive Coordinate Calibration Notebook

Paste and run the following code in a Jupyter Notebook cell. It provides:

1. A dropdown to select which spot to tweak (e.g., handicapped vs. standard spots).
2. Live sliders for `X`, `Y`, `Width`, and `Height` that immediately re-draw the box on the canvas.
3. A **Save to JSON** button so your batch-processing scripts can load the verified coordinates directly.

In [ ]:
import json
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Load your reference screenshot
IMAGE_PATH = "sample_screenshot.png"  # Path to one representative screenshot
img = Image.open(IMAGE_PATH).convert("RGB")
img_w, img_h = img.size

# 2. Initialize default coordinates (Rough estimates to start with)
# We initialize 20 spots. Standard spots get standard sizes; handicapped can be tweaked.
DEFAULT_W, DEFAULT_H = 80, 110
spots_config = {}

# Seed initial rough positions (e.g., 2 rows of 10)
spot_id = 1
for row in range(2):
    for col in range(10):
        name = f"spot_{spot_id:02d}"
        # Default regular spacing
        spots_config[name] = {
            "x": int(100 + col * 95),
            "y": int(200 + row * 130),
            "w": DEFAULT_W,
            "h": DEFAULT_H,
            "type": "handicapped" if spot_id in [1, 2] else "standard"
        }
        spot_id += 1

# If a previously saved config exists, load it
CONFIG_FILE = Path("spots_calibrated_config.json")
if CONFIG_FILE.exists():
    with open(CONFIG_FILE, "r") as f:
        spots_config = json.load(f)

# ---------------------------------------------------------------------------
# 3. Interactive UI Controls
# ---------------------------------------------------------------------------
spot_selector = widgets.Dropdown(
    options=list(spots_config.keys()),
    value="spot_01",
    description="Select Spot:"
)

x_slider = widgets.IntSlider(min=0, max=img_w, step=1, description="X Position:")
y_slider = widgets.IntSlider(min=0, max=img_h, step=1, description="Y Position:")
w_slider = widgets.IntSlider(min=10, max=400, step=1, description="Width:")
h_slider = widgets.IntSlider(min=10, max=400, step=1, description="Height:")
spot_type_toggle = widgets.ToggleButtons(
    options=["standard", "handicapped"],
    description="Type:"
)

save_btn = widgets.Button(description="Save Config to JSON", button_style="success", icon="save")
status_label = widgets.Label(value="")

def sync_sliders_with_spot(spot_name):
    cfg = spots_config[spot_name]
    x_slider.value = cfg["x"]
    y_slider.value = cfg["y"]
    w_slider.value = cfg["w"]
    h_slider.value = cfg["h"]
    spot_type_toggle.value = cfg.get("type", "standard")

sync_sliders_with_spot(spot_selector.value)

# ---------------------------------------------------------------------------
# 4. Canvas Renderer
# ---------------------------------------------------------------------------
plot_output = widgets.Output()

def render_canvas():
    with plot_output:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(14, 8), dpi=100)
        ax.imshow(img)

        active_spot = spot_selector.value

        for name, cfg in spots_config.items():
            is_active = (name == active_spot)
            is_handicapped = (cfg.get("type") == "handicapped")

            # Determine colors: Active spot gets highlighted in bright cyan
            if is_active:
                edge_color = "#00FFFF"
                lw = 2.5
            elif is_handicapped:
                edge_color = "#FFA500"  # Orange for handicapped
                lw = 1.5
            else:
                edge_color = "#00FF00"  # Green for standard
                lw = 1.2

            rect = patches.Rectangle(
                (cfg["x"], cfg["y"]), cfg["w"], cfg["h"],
                linewidth=lw, edgecolor=edge_color, facecolor="none"
            )
            ax.add_patch(rect)

            ax.text(
                cfg["x"], max(0, cfg["y"] - 4),
                name, color="black", fontsize=8, fontweight="bold",
                bbox=dict(facecolor=edge_color, edgecolor="none", pad=1.5, alpha=0.85)
            )

        ax.set_title(f"Editing: {active_spot} ({spots_config[active_spot]['type'].upper()})", fontsize=12, fontweight="bold")
        ax.axis("off")
        plt.tight_layout()
        plt.show()

# ---------------------------------------------------------------------------
# 5. Event Callbacks
# ---------------------------------------------------------------------------
def on_spot_change(change):
    sync_sliders_with_spot(change["new"])
    render_canvas()

def on_param_change(change):
    active = spot_selector.value
    spots_config[active]["x"] = x_slider.value
    spots_config[active]["y"] = y_slider.value
    spots_config[active]["w"] = w_slider.value
    spots_config[active]["h"] = h_slider.value
    spots_config[active]["type"] = spot_type_toggle.value
    render_canvas()

def on_save_clicked(b):
    with open(CONFIG_FILE, "w") as f:
        json.dump(spots_config, f, indent=2)
    status_label.value = f"Saved {len(spots_config)} coordinates to {CONFIG_FILE.name}!"

spot_selector.observe(on_spot_change, names="value")
x_slider.observe(on_param_change, names="value")
y_slider.observe(on_param_change, names="value")
w_slider.observe(on_param_change, names="value")
h_slider.observe(on_param_change, names="value")
spot_type_toggle.observe(on_param_change, names="value")
save_btn.on_click(on_save_clicked)

# Display controls
controls_box = widgets.VBox([
    widgets.HBox([spot_selector, spot_type_toggle]),
    widgets.HBox([x_slider, y_slider]),
    widgets.HBox([w_slider, h_slider]),
    widgets.HBox([save_btn, status_label])
])

display(controls_box, plot_output)
render_canvas()

---

### Step 3: Fast Calibration Workflow

1. **Pick the Irregular Spaces First:** Use the dropdown to select the handicapped spaces, toggle the type to `handicapped`, and adjust `Width`, `Height`, and `Position` until the boundaries enclose the icon or vehicle stall cleanly.
2. **Review Neighbors:** As you shift the irregular spaces, the orange rectangle visually distinguishes them from standard green spaces.
3. **Persist:** Click **Save Config to JSON**. It produces a structured map like:
```json
{
  "spot_01": { "x": 105, "y": 210, "w": 120, "h": 110, "type": "handicapped" },
  "spot_02": { "x": 235, "y": 210, "w": 80,  "h": 110, "type": "standard" }
}

```



---

### Step 4: Loading into Your Multiprocessing Script

In your high-throughput processing script for the 6,916 images, replace the hardcoded math loop with the generated JSON configuration:

In [ ]:
import json
from pathlib import Path

# Load calibrated coordinates
with open("spots_calibrated_config.json") as f:
    spots_data = json.load(f)

# Format into (xmin, ymin, xmax, ymax) lookup dictionary
SPOTS = {
    name: (cfg["x"], cfg["y"], cfg["x"] + cfg["w"], cfg["y"] + cfg["h"])
    for name, cfg in spots_data.items()
}

This decouples the visual layout inspection from your pipeline logic and handles varying stall sizes without breaking downstream code.

In [1]:
from pathlib import Path

# Base root (current directory)
ROOT = Path(".")

# 1. Directories to create
DIRECTORIES = [
    ROOT / "configs",
    ROOT / "data" / "raw_screenshots",
    ROOT / "data" / "processed",
    ROOT / "data" / "models",
    ROOT / "notebooks",
    ROOT / "src" / "ev_occupancy" / "ingestion",
    ROOT / "src" / "ev_occupancy" / "classical",
    ROOT / "src" / "ev_occupancy" / "vision",
    ROOT / "src" / "ev_occupancy" / "semantic",
    ROOT / "src" / "ev_occupancy" / "serving",
    ROOT / "deploy",
    ROOT / "tests",
]

# 2. Files with starter content
FILES = {
    ROOT / ".gitignore": """# Byte-compiled / optimized / DLL files
__pycache__/
*.py[cod]
*$py.class
*.ipynb_checkpoints/

# Virtual environments
.venv/
env/
venv/
ENV/

# Data and models (Track folders via .gitkeep, ignore contents)
data/raw_screenshots/*
!data/raw_screenshots/.gitkeep
data/processed/*
!data/processed/.gitkeep
data/models/*
!data/models/.gitkeep

# Data artifacts
*.parquet
*.h5
*.onnx
*.pt
*.pth
*.pkl

# Secrets / local overrides
.env
.env.local
""",
    ROOT / "data" / "raw_screenshots" / ".gitkeep": "",
    ROOT / "data" / "processed" / ".gitkeep": "",
    ROOT / "data" / "models" / ".gitkeep": "",
    ROOT / "src" / "ev_occupancy" / "__init__.py": '"""EV Occupancy Monitoring and Forecasting Module."""\n',
    ROOT / "src" / "ev_occupancy" / "ingestion" / "__init__.py": "",
    ROOT / "src" / "ev_occupancy" / "classical" / "__init__.py": "",
    ROOT / "src" / "ev_occupancy" / "vision" / "__init__.py": "",
    ROOT / "src" / "ev_occupancy" / "semantic" / "__init__.py": "",
    ROOT / "src" / "ev_occupancy" / "serving" / "__init__.py": "",
    ROOT / "tests" / "__init__.py": "",
}


def build_scaffold():
    print(f"Scaffolding repository layout in: {ROOT.resolve()}")

    # Create directories
    for directory in DIRECTORIES:
        directory.mkdir(parents=True, exist_ok=True)
        print(f"  [dir]  {directory}")

    # Create files
    for file_path, content in FILES.items():
        if not file_path.exists():
            file_path.write_text(content, encoding="utf-8")
            print(f"  [file] {file_path}")
        else:
            print(f"  [skip] {file_path} (already exists)")

    print("\nProject scaffold created successfully.")


if __name__ == "__main__":
    build_scaffold()

Scaffolding repository layout in: /content
  [dir]  configs
  [dir]  data/raw_screenshots
  [dir]  data/processed
  [dir]  data/models
  [dir]  notebooks
  [dir]  src/ev_occupancy/ingestion
  [dir]  src/ev_occupancy/classical
  [dir]  src/ev_occupancy/vision
  [dir]  src/ev_occupancy/semantic
  [dir]  src/ev_occupancy/serving
  [dir]  deploy
  [dir]  tests
  [file] .gitignore
  [file] data/raw_screenshots/.gitkeep
  [file] data/processed/.gitkeep
  [file] data/models/.gitkeep
  [file] src/ev_occupancy/__init__.py
  [file] src/ev_occupancy/ingestion/__init__.py
  [file] src/ev_occupancy/classical/__init__.py
  [file] src/ev_occupancy/vision/__init__.py
  [file] src/ev_occupancy/semantic/__init__.py
  [file] src/ev_occupancy/serving/__init__.py
  [file] tests/__init__.py

Project scaffold created successfully.
